In [1]:
import sys
import os
import yfinance as yf
import pandas as pd
import plotly.graph_objects as go
import sys
from pathlib import Path

In [2]:

# Ruta raíz del proyecto
project_path =  r"C:\Users\Gisela\Documents\GitHub\Kronos" #os.path.abspath("Kronos") 

# Agregar al path
sys.path.append(project_path)

print(project_path)
# print(sys.path)

C:\Users\Gisela\Documents\GitHub\Kronos


# Datos

In [3]:
import yfinance as yf

df = yf.download("PFAVAL.CL",
                 start="2025-01-01",
                 end="2026-05-28")


df.to_csv("pfaval.csv")

df.head()

[*********************100%***********************]  1 of 1 completed


Price,Close,High,Low,Open,Volume
Ticker,PFAVAL.CL,PFAVAL.CL,PFAVAL.CL,PFAVAL.CL,PFAVAL.CL
Date,,,,,
2025-01-02,433.452271,433.452271,423.988247,422.095442,1242177
2025-01-03,438.184326,440.077131,434.398716,433.452314,2385850
2025-01-07,440.077179,440.077179,430.613154,438.184374,1979583
2025-01-08,438.184326,442.916338,435.345119,440.077131,564204
2025-01-09,435.345093,439.130702,435.345093,438.184300,3706319


In [4]:
# Eliminar filas multi index
df.columns = df.columns.get_level_values(0)
df.head()

Price,Close,High,Low,Open,Volume
Date,,,,,
2025-01-02,433.452271,433.452271,423.988247,422.095442,1242177
2025-01-03,438.184326,440.077131,434.398716,433.452314,2385850
2025-01-07,440.077179,440.077179,430.613154,438.184374,1979583
2025-01-08,438.184326,442.916338,435.345119,440.077131,564204
2025-01-09,435.345093,439.130702,435.345093,438.184300,3706319


In [5]:
print('Cols y filas en df: ',df.shape)
print('Nombre de cols: ',df.columns)

Cols y filas en df:  (350, 5)
Nombre de cols:  Index(['Close', 'High', 'Low', 'Open', 'Volume'], dtype='str', name='Price')


In [6]:
# Vertificar duplicados en el índice
df.index.duplicated().sum()

np.int64(0)

In [7]:
df.columns.name = None

df = df.reset_index()

In [8]:
df.head()

,Date,Close,High,Low,Open,Volume
0,2025-01-02,433.452271,433.452271,423.988247,422.095442,1242177
1,2025-01-03,438.184326,440.077131,434.398716,433.452314,2385850
2,2025-01-07,440.077179,440.077179,430.613154,438.184374,1979583
3,2025-01-08,438.184326,442.916338,435.345119,440.077131,564204
4,2025-01-09,435.345093,439.130702,435.345093,438.184300,3706319


In [9]:
# Convertir index a timestamp y renombrar cols

# df['index'] = pd.to_datetime(df['index'])
df.columns = ['date', 'close', 'high', 'low', 'open', 'volume']



In [10]:

df.head()

,date,close,high,low,open,volume
0,2025-01-02,433.452271,433.452271,423.988247,422.095442,1242177
1,2025-01-03,438.184326,440.077131,434.398716,433.452314,2385850
2,2025-01-07,440.077179,440.077179,430.613154,438.184374,1979583
3,2025-01-08,438.184326,442.916338,435.345119,440.077131,564204
4,2025-01-09,435.345093,439.130702,435.345093,438.184300,3706319


# Preparación de los datos para el modelo

In [11]:
# Defnir ventana de contexto y predicción

lookback = 343
pred_len = 7

# Prepare inputs for the predictor
x_df = df.loc[:lookback-1, ['open', 'high', 'low', 'close', 'volume']]
x_timestamp = df.loc[:lookback-1, 'date']
y_timestamp = df.loc[lookback:lookback+pred_len-1, 'date']

In [12]:
import sys
print(sys.executable)
import safetensors
print(safetensors.__version__)

c:\Users\Gisela\Documents\GitHub\Talleres_IA_2026\.venv\Scripts\python.exe
0.7.0


In [13]:
from Kronos.model.kronos import Kronos, KronosTokenizer, KronosPredictor

# Cargar desde HF
tokenizer = KronosTokenizer.from_pretrained("NeoQuasar/Kronos-Tokenizer-base")
model = Kronos.from_pretrained("NeoQuasar/Kronos-small")

c:\Users\Gisela\Documents\GitHub\Talleres_IA_2026\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [14]:
# Inicializar predictor
predictor = KronosPredictor(model, tokenizer, max_context= 512)

In [15]:
# Generar predicciones
pred_df = predictor.predict(
    df=x_df,
    x_timestamp=x_timestamp,
    y_timestamp=y_timestamp,
    pred_len=pred_len,
    T=1.0,          # Temperature for sampling
    top_p=0.9,      # Nucleus sampling probability
    sample_count=1  # Number of forecast paths to generate and average
)

print("Forecasted Data Head:")
print(pred_df.head())

100%|██████████| 7/7 [00:00<00:00, 11.18it/s]

Forecasted Data Head:
                  open        high         low       close      volume  \
date                                                                     
2026-05-19  771.763000  792.582275  767.381348  787.147217  2338798.25   
2026-05-20  780.557251  785.689697  769.574585  784.270996  5589213.50   
2026-05-21  776.622375  802.125488  764.794678  772.298706  2954572.50   
2026-05-22  776.951477  776.307617  745.510864  751.247314  4297629.00   
2026-05-25  756.985840  776.507935  750.999695  770.940918  1673990.50   

                  amount  
date                      
2026-05-19  1.502264e+09  
2026-05-20  3.900896e+09  
2026-05-21  1.964312e+09  
2026-05-22  2.952979e+09  
2026-05-25  1.026639e+09  


In [16]:
pred_df.tail(7)

,open,high,low,close,volume,amount
date,,,,,,
2026-05-19,771.763000,792.582275,767.381348,787.147217,2338798.250,1.502264e+09
2026-05-20,780.557251,785.689697,769.574585,784.270996,5589213.500,3.900896e+09
2026-05-21,776.622375,802.125488,764.794678,772.298706,2954572.500,1.964312e+09
2026-05-22,776.951477,776.307617,745.510864,751.247314,4297629.000,2.952979e+09
2026-05-25,756.985840,776.507935,750.999695,770.940918,1673990.500,1.026639e+09
2026-05-26,768.941101,779.747070,759.589661,774.238708,3498151.000,2.342190e+09
2026-05-27,775.562500,790.818481,768.891663,786.085449,1818149.375,1.137917e+09


In [17]:
# Función para Visualizar
import plotly.graph_objects as go

def plot_prediction(df, pred_df):
    pred_df.index = df.index[-pred_df.shape[0]:]
    close_df = pd.concat([df['close'], pred_df['close']], axis=1)
    close_df.columns = ['Ground Truth', 'Prediction']

    volume_df = pd.concat([df['volume'], pred_df['volume']], axis=1)
    volume_df.columns = ['Ground Truth', 'Prediction']

    fig = go.Figure()

    fig.add_trace(go.Scatter(
        x=close_df.index,
        y=close_df['Ground Truth'],
        mode='lines',
        name='Close Ground Truth',
        line=dict(color='blue', width=2)
    ))
    fig.add_trace(go.Scatter(
        x=close_df.index,
        y=close_df['Prediction'],
        mode='lines',
        name='Close Prediction',
        line=dict(color='red', width=2, dash='dash')
    ))

    fig.update_layout(
        title='Predicción de Precio de Cierre',
        xaxis_title='Fecha',
        yaxis_title='Precio de Cierre',
        legend_title='Serie',
        template='plotly_white'
    )

    fig_volume = go.Figure()

    fig_volume.add_trace(go.Bar(
        x=volume_df.index,
        y=volume_df['Ground Truth'],
        name='Volume Ground Truth',
        marker_color='blue',
        opacity=0.6
    ))
    fig_volume.add_trace(go.Bar(
        x=volume_df.index,
        y=volume_df['Prediction'],
        name='Volume Prediction',
        marker_color='red',
        opacity=0.6
    ))

    fig_volume.update_layout(
        title='Predicción de Volumen',
        xaxis_title='Fecha',
        yaxis_title='Volumen',
        barmode='group',
        template='plotly_white'
    )

    fig.show()
    fig_volume.show()

In [18]:
# 5. Visualize Results
print("Forecasted Data Head:")
print(pred_df.head())

# Combine historical and forecasted data for plotting
df = df.loc[:lookback+pred_len-1]

# visualize
plot_prediction(df, pred_df)

Forecasted Data Head:
                  open        high         low       close      volume  \
date                                                                     
2026-05-19  771.763000  792.582275  767.381348  787.147217  2338798.25   
2026-05-20  780.557251  785.689697  769.574585  784.270996  5589213.50   
2026-05-21  776.622375  802.125488  764.794678  772.298706  2954572.50   
2026-05-22  776.951477  776.307617  745.510864  751.247314  4297629.00   
2026-05-25  756.985840  776.507935  750.999695  770.940918  1673990.50   

                  amount  
date                      
2026-05-19  1.502264e+09  
2026-05-20  3.900896e+09  
2026-05-21  1.964312e+09  
2026-05-22  2.952979e+09  
2026-05-25  1.026639e+09  


In [19]:
df.tail()

,date,close,high,low,open,volume
345,2026-05-21,777.390137,782.373407,764.433634,776.393483,610851
346,2026-05-22,776.393494,785.363380,764.433645,777.390148,656376
347,2026-05-25,789.349976,791.343284,771.410203,776.393473,722432
348,2026-05-26,844.000000,844.000000,790.000000,792.000000,2450232
349,2026-05-27,840.000000,852.000000,834.000000,844.000000,2742929
